## Step 1: Install Required Packages

First, we need to install CrewAI and its dependencies. We'll use:
- `crewai`: The main framework
- `crewai-tools`: Pre-built tools for agents

In [1]:
# Install required packages
!pip install -q crewai crewai-tools

## Step 2: Setup API Keys and Environment

To use OpenAI's GPT-4.1-mini model, you need an API key and base URL.

**Security Best Practice:** Store your API key in a `.env` file, not in the code!

In [2]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

if "OPENAI_BASE_URL" not in os.environ:
    os.environ["OPENAI_BASE_URL"] = getpass.getpass("Enter your OpenAI Base URL: ")

## Step 3: Create Custom Tools

Tools are functions that agents can use to perform specific tasks. Let's create custom tools for our movie production assistant.

### Tool 1: Script Sentiment Analyzer
Analyzes the emotional tone of a script excerpt.

In [3]:
from crewai.tools import tool
import re

@tool("Script Sentiment Analyzer")
def analyze_script_sentiment(script_text: str) -> str:
    """
    Analyzes the sentiment and emotional tone of a movie script excerpt.
    Returns sentiment score and dominant emotions.

    Args:
        script_text: A string containing the script excerpt to analyze
    """
    # Simple sentiment analysis based on keywords
    positive_words = ['love', 'happy', 'joy', 'wonderful', 'amazing', 'brilliant', 'triumph', 'success']
    negative_words = ['hate', 'sad', 'angry', 'terrible', 'awful', 'tragedy', 'failure', 'death']
    suspense_words = ['mystery', 'unknown', 'hidden', 'secret', 'suspense', 'thriller']

    script_lower = script_text.lower()

    pos_count = sum(1 for word in positive_words if word in script_lower)
    neg_count = sum(1 for word in negative_words if word in script_lower)
    sus_count = sum(1 for word in suspense_words if word in script_lower)

    total = pos_count + neg_count + sus_count

    if total == 0:
        return "Sentiment: Neutral - No strong emotional indicators detected."

    result = f"Sentiment Analysis Results:\n"
    result += f"- Positive tone: {pos_count}/{total} ({pos_count/total*100:.1f}%)\n"
    result += f"- Negative tone: {neg_count}/{total} ({neg_count/total*100:.1f}%)\n"
    result += f"- Suspenseful tone: {sus_count}/{total} ({sus_count/total*100:.1f}%)\n"

    if pos_count > neg_count and pos_count > sus_count:
        result += "\nOverall: Uplifting and positive story"
    elif neg_count > pos_count and neg_count > sus_count:
        result += "\nOverall: Dark and dramatic narrative"
    else:
        result += "\nOverall: Tense and mysterious atmosphere"

    return result

# Test the tool
test_script = "A tale of love and triumph as Sarah discovers the hidden mystery that brings her joy."
print(analyze_script_sentiment.run(test_script))

Sentiment Analysis Results:
- Positive tone: 3/5 (60.0%)
- Negative tone: 0/5 (0.0%)
- Suspenseful tone: 2/5 (40.0%)

Overall: Uplifting and positive story


### Tool 2: Budget Calculator
Calculates production costs based on different categories.

In [4]:
@tool("Production Budget Calculator")
def calculate_production_budget(category: str, days: int = 30, crew_size: int = 50) -> str:
    """
    Calculates estimated production budget for different movie categories.

    Args:
        category: Type of production (indie, medium, blockbuster)
        days: Number of shooting days
        crew_size: Number of crew members
    """
    # Base daily rates per category
    rates = {
        'indie': {'daily': 50000, 'crew': 500, 'equipment': 10000},
        'medium': {'daily': 200000, 'crew': 2000, 'equipment': 50000},
        'blockbuster': {'daily': 1000000, 'crew': 10000, 'equipment': 250000}
    }

    category = category.lower()
    if category not in rates:
        return f"Unknown category. Choose from: {', '.join(rates.keys())}"

    rate = rates[category]

    production_cost = rate['daily'] * days
    crew_cost = rate['crew'] * crew_size * days
    equipment_cost = rate['equipment'] * days
    contingency = (production_cost + crew_cost + equipment_cost) * 0.15

    total = production_cost + crew_cost + equipment_cost + contingency

    result = f"📊 Budget Breakdown for {category.upper()} Production:\n"
    result += f"\n🎬 Production Costs: ${production_cost:,}"
    result += f"\n👥 Crew Costs ({crew_size} members): ${crew_cost:,}"
    result += f"\n📹 Equipment Rental: ${equipment_cost:,}"
    result += f"\n🛡️ Contingency (15%): ${contingency:,}"
    result += f"\n\n💰 TOTAL ESTIMATED BUDGET: ${total:,}"
    result += f"\n📅 For {days} shooting days"

    return result

# Test the tool
print(calculate_production_budget.run("medium", days=45, crew_size=75))

📊 Budget Breakdown for MEDIUM Production:

🎬 Production Costs: $9,000,000
👥 Crew Costs (75 members): $6,750,000
📹 Equipment Rental: $2,250,000
🛡️ Contingency (15%): $2,700,000.0

💰 TOTAL ESTIMATED BUDGET: $20,700,000.0
📅 For 45 shooting days


### Tool 3: Soundtrack Genre Recommender
Suggests music genres based on the movie's theme and mood.

In [5]:
@tool("Soundtrack Genre Recommender")
def recommend_soundtrack_genre(movie_genre: str, mood: str) -> str:
    """
    Recommends soundtrack genres based on movie genre and mood.

    Args:
        movie_genre: The genre of the movie (action, drama, comedy, horror, sci-fi, romance)
        mood: The desired mood (intense, light, emotional, mysterious, upbeat)
    """
    recommendations = {
        'action': {
            'intense': ['Epic Orchestral', 'Electronic/Synth', 'Heavy Metal'],
            'light': ['Pop Rock', 'Funk', 'Electronic Pop'],
            'emotional': ['Cinematic Strings', 'Piano & Strings', 'Alternative Rock'],
        },
        'drama': {
            'intense': ['Classical', 'Dramatic Orchestral', 'Jazz Noir'],
            'emotional': ['Piano Solo', 'String Quartet', 'Acoustic Guitar'],
            'mysterious': ['Ambient', 'Minimalist Piano', 'Contemporary Classical'],
        },
        'comedy': {
            'upbeat': ['Jazz', 'Pop', 'Ska', 'Funk'],
            'light': ['Acoustic Pop', 'Ukulele', 'Whistling & Bells'],
        },
        'horror': {
            'intense': ['Dark Ambient', 'Industrial', 'Atonal Orchestral'],
            'mysterious': ['Theremin', 'Dissonant Strings', 'Electronic Drone'],
        },
        'sci-fi': {
            'intense': ['Electronic/Synth', 'Cyberpunk', 'Orchestral Hybrid'],
            'mysterious': ['Ambient Electronic', 'Experimental', 'Synth Pad'],
        },
        'romance': {
            'emotional': ['Classical Piano', 'Acoustic', 'Indie Folk'],
            'light': ['Soft Pop', 'Bossa Nova', 'Acoustic Guitar'],
            'upbeat': ['Pop', 'Soul', 'Light Jazz'],
        }
    }

    genre = movie_genre.lower()
    mood_key = mood.lower()

    if genre not in recommendations:
        return f"Unknown genre. Try: {', '.join(recommendations.keys())}"

    genre_moods = recommendations[genre]
    if mood_key not in genre_moods:
        return f"For {genre}, available moods are: {', '.join(genre_moods.keys())}"

    suggested = genre_moods[mood_key]

    result = f"🎵 Soundtrack Recommendations for {genre.upper()} ({mood} mood):\n\n"
    for i, genre_rec in enumerate(suggested, 1):
        result += f"{i}. {genre_rec}\n"

    result += f"\n💡 Tip: Consider mixing {suggested[0]} with {suggested[-1]} for dynamic scenes!"

    return result

# Test the tool
print(recommend_soundtrack_genre.run("sci-fi", "mysterious"))

🎵 Soundtrack Recommendations for SCI-FI (mysterious mood):

1. Ambient Electronic
2. Experimental
3. Synth Pad

💡 Tip: Consider mixing Ambient Electronic with Synth Pad for dynamic scenes!


## Step 4: Integrate MCP Tool (Model Context Protocol)

MCP tools allow agents to interact with external systems and APIs. Here we'll create a simulated MCP tool for accessing a movie database.

**Note:** In production, MCP tools connect to real external services. This is a simplified demonstration.

In [6]:
from typing import Dict, Any

@tool("Movie Market Research MCP")
def fetch_market_data(genre: str) -> str:
    """
    Fetches market research data for a specific movie genre from the database.
    This simulates an MCP tool that connects to external market research APIs.

    Args:
        genre: The movie genre to research (action, drama, comedy, horror, sci-fi, romance)
    """
    # Simulated market data (in production, this would call a real API)
    market_database = {
        'action': {
            'avg_box_office': '$450M',
            'audience_demographic': '18-35 years, 65% male',
            'peak_season': 'Summer (May-August)',
            'trending_themes': ['Superheroes', 'International espionage', 'Disaster scenarios'],
            'streaming_performance': 'High - 85% completion rate',
        },
        'drama': {
            'avg_box_office': '$120M',
            'audience_demographic': '30-60 years, 55% female',
            'peak_season': 'Awards season (October-February)',
            'trending_themes': ['Social justice', 'Biography', 'Family dynamics'],
            'streaming_performance': 'Medium - 72% completion rate',
        },
        'comedy': {
            'avg_box_office': '$180M',
            'audience_demographic': '18-45 years, balanced gender',
            'peak_season': 'Year-round, slight spike in holidays',
            'trending_themes': ['Romantic comedy', 'Workplace humor', 'Cultural clash'],
            'streaming_performance': 'Very High - 90% completion rate',
        },
        'horror': {
            'avg_box_office': '$95M',
            'audience_demographic': '16-30 years, 52% male',
            'peak_season': 'Fall (September-November)',
            'trending_themes': ['Psychological horror', 'Folk horror', 'Home invasion'],
            'streaming_performance': 'High - 82% completion rate',
        },
        'sci-fi': {
            'avg_box_office': '$380M',
            'audience_demographic': '18-40 years, 60% male',
            'peak_season': 'Summer and Holiday season',
            'trending_themes': ['AI & technology', 'Space exploration', 'Time travel'],
            'streaming_performance': 'High - 88% completion rate',
        },
        'romance': {
            'avg_box_office': '$85M',
            'audience_demographic': '18-50 years, 70% female',
            'peak_season': 'Valentine\'s Day season (January-February)',
            'trending_themes': ['Second chance romance', 'Holiday romance', 'Cross-cultural love'],
            'streaming_performance': 'Very High - 92% completion rate',
        },
    }

    genre_key = genre.lower()

    if genre_key not in market_database:
        return f"❌ No market data available for '{genre}'. Available genres: {', '.join(market_database.keys())}"

    data = market_database[genre_key]

    result = f"📈 Market Research Report: {genre.upper()} Genre\n"
    result += f"\n💰 Average Box Office: {data['avg_box_office']}"
    result += f"\n👥 Target Demographic: {data['audience_demographic']}"
    result += f"\n📅 Best Release Window: {data['peak_season']}"
    result += f"\n🎬 Trending Themes:"
    for theme in data['trending_themes']:
        result += f"\n   • {theme}"
    result += f"\n📺 Streaming Performance: {data['streaming_performance']}"
    result += f"\n\n✅ Data retrieved from Market Intelligence Database (MCP)"

    return result

# Test the MCP tool
print(fetch_market_data.run("action"))

📈 Market Research Report: ACTION Genre

💰 Average Box Office: $450M
👥 Target Demographic: 18-35 years, 65% male
📅 Best Release Window: Summer (May-August)
🎬 Trending Themes:
   • Superheroes
   • International espionage
   • Disaster scenarios
📺 Streaming Performance: High - 85% completion rate

✅ Data retrieved from Market Intelligence Database (MCP)


## Step 5: Configure the Language Model

Now let's set up OpenAI's GPT-4.1-mini model to power our agents. This model is fast and cost-effective, perfect for agent-based workflows.

In [7]:
from crewai import LLM

# Initialize GPT-4.1-mini using CrewAI's LLM wrapper
llm = LLM(
    model="openai/gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
    temperature=0.7,  # Controls creativity (0=focused, 1=creative)
)

print("✅ Language model configured: GPT-4.1-mini")
print("   - Model: openai/gpt-4.1-mini")
print("   - Temperature: 0.7 (balanced creativity)")

✅ Language model configured: GPT-4.1-mini
   - Model: openai/gpt-4.1-mini
   - Temperature: 0.7 (balanced creativity)


## Step 6: Create Specialized Agents

Agents are AI assistants with specific roles and tools. Let's create a crew of specialized agents for movie production:

1. **Script Analyst** - Analyzes scripts and provides creative feedback
2. **Budget Manager** - Handles financial planning and budgeting
3. **Market Researcher** - Provides market insights and recommendations

In [8]:
from crewai import Agent

# Agent 1: Script Analyst
script_analyst = Agent(
    role="Script Analyst",
    goal="Analyze movie scripts for sentiment, themes, and recommend appropriate soundtracks",
    backstory="""You are an experienced script analyst with 15 years in Hollywood.
    You have a keen eye for emotional depth and understand how music enhances storytelling.
    You've worked on numerous award-winning films and know what makes a script compelling.""",
    tools=[analyze_script_sentiment, recommend_soundtrack_genre],
    llm=llm,
    verbose=True,  # Shows detailed thinking process
    allow_delegation=False,  # Can't delegate tasks to other agents
)

# Agent 2: Budget Manager
budget_manager = Agent(
    role="Production Budget Manager",
    goal="Calculate accurate production budgets and optimize spending across departments",
    backstory="""You are a meticulous financial expert specializing in film production.
    With an MBA and 10 years of experience managing budgets for indie to blockbuster films,
    you ensure projects stay financially viable while maintaining creative vision.""",
    tools=[calculate_production_budget],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Agent 3: Market Researcher
market_researcher = Agent(
    role="Film Market Research Analyst",
    goal="Provide data-driven insights about market trends, audience preferences, and optimal release strategies",
    backstory="""You are a data scientist specializing in entertainment industry analytics.
    You analyze box office trends, streaming data, and audience demographics to help studios
    make informed decisions about production and distribution.""",
    tools=[fetch_market_data],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Created 3 specialized agents:")
print("   1. Script Analyst (sentiment & soundtrack)")
print("   2. Budget Manager (financial planning)")
print("   3. Market Researcher (market intelligence)")

✅ Created 3 specialized agents:
   1. Script Analyst (sentiment & soundtrack)
   2. Budget Manager (financial planning)
   3. Market Researcher (market intelligence)


## Step 7: Define Tasks for Agents

Tasks are specific assignments given to agents. Each task has:
- A clear description
- An assigned agent
- Expected output format

In [9]:
from crewai import Task

# Sample movie concept for our tasks
movie_concept = """
Title: "Echoes of Tomorrow"
Genre: Sci-Fi Thriller
Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals
humanity's tragic future. Racing against time and a mysterious organization, she must decide
whether to alter the timeline or accept humanity's fate.

Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both
triumph and terror. The hidden secret she uncovers could save millions, but the mystery
of who left this message haunts her every waking moment."
"""

# Task 1: Analyze the script
task_analyze_script = Task(
    description=f"""Analyze the following movie concept and script excerpt:

    {movie_concept}

    Use your tools to:
    1. Analyze the sentiment of the script excerpt
    2. Recommend appropriate soundtrack genres based on the genre and mood
    3. Provide creative suggestions for enhancing the emotional impact
    """,
    expected_output="A detailed analysis including sentiment breakdown, soundtrack recommendations, and creative suggestions",
    agent=script_analyst,
)

# Task 2: Calculate budget
task_budget_calculation = Task(
    description="""Based on the sci-fi thriller movie concept, calculate the production budget.

    Assume:
    - This is a MEDIUM budget production
    - 60 days of shooting
    - Crew size of 85 people

    Provide a detailed breakdown and recommendations for budget optimization.
    """,
    expected_output="Complete budget breakdown with optimization recommendations",
    agent=budget_manager,
)

# Task 3: Market research
task_market_research = Task(
    description="""Conduct market research for the sci-fi thriller genre.

    Use the MCP tool to fetch market data and provide:
    1. Current market performance of sci-fi films
    2. Target audience insights
    3. Optimal release timing
    4. Strategic recommendations based on trending themes
    """,
    expected_output="Comprehensive market analysis with strategic release recommendations",
    agent=market_researcher,
)

print("✅ Created 3 tasks:")
print("   1. Script Analysis Task")
print("   2. Budget Calculation Task")
print("   3. Market Research Task")

✅ Created 3 tasks:
   1. Script Analysis Task
   2. Budget Calculation Task
   3. Market Research Task


## Step 8: Assemble the Crew and Execute

Now we bring it all together! A Crew coordinates multiple agents working on related tasks.

**Process Types:**
- `sequential`: Tasks execute one after another (used here)
- `hierarchical`: A manager agent coordinates other agents

In [18]:
os.environ["CREWAI_TRACING_ENABLED"] = "true"

In [19]:
from crewai import Crew, Process

# Assemble the crew
movie_production_crew = Crew(
    agents=[script_analyst, budget_manager, market_researcher],
    tasks=[task_analyze_script, task_budget_calculation, task_market_research],
    process=Process.sequential,  # Tasks run in order
    verbose=True,  # Enable detailed output for learning
)

print("✅ Crew assembled with 3 agents and 3 tasks")
print("\n🚀 Starting production analysis...\n")
print("=" * 80)

# Execute the crew's tasks
result = movie_production_crew.kickoff()

print("\n" + "=" * 80)
print("\n✅ Analysis Complete!\n")

✅ Crew assembled with 3 agents and 3 tasks

🚀 Starting production analysis...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8935cf36-49e7-448a-84b4-38ff43191ae6                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following movie concept and script excerpt:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: "Echoes of Tomorrow"                                                                                    │
│  Genre: Sci-Fi Thriller                                                                                         │
│  Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals                     │
│  humanity's tragic future. Racing against time and a mysterious organization, she must decide                   │
│  whether to alter the timeline or accept humanity's fate.                                                       │
│                                                                                                                 │
│  Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both                        │
│  triumph and terror. The hidden secret she uncovers could save millions, but the mystery                        │
│  of who left this message haunts her every waking moment."                                                      │
│                                                                                                                 │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the sentiment of the script excerpt                                                             │
│      2. Recommend appropriate soundtrack genres based on the genre and mood                                     │
│      3. Provide creative suggestions for enhancing the emotional impact                                         │
│                                                                                                                 │
│  ID: ce46c211-2368-46fa-8331-53ba9d5b2a47                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Task: Analyze the following movie concept and script excerpt:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: "Echoes of Tomorrow"                                                                                    │
│  Genre: Sci-Fi Thriller                                                                                         │
│  Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals                     │
│  humanity's tragic future. Racing against time and a mysterious organization, she must decide                   │
│  whether to alter the timeline or accept humanity's fate.                                                       │
│                                                                                                                 │
│  Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both                        │
│  triumph and terror. The hidden secret she uncovers could save millions, but the mystery                        │
│  of who left this message haunts her every waking moment."                                                      │
│                                                                                                                 │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the sentiment of the script excerpt                                                             │
│      2. Recommend appropriate soundtrack genres based on the genre and mood                                     │
│      3. Provide creative suggestions for enhancing the emotional impact                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool script_sentiment_analyzer executed with result: Sentiment Analysis Results:
- Positive tone: 1/5 (20.0%)
- Negative tone: 0/5 (0.0%)
- Suspenseful tone: 4/5 (80.0%)

Overall: Tense and mysterious atmosphere...


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: movie_market_research_mcp                                                                                │
│  Output: 📈 Market Research Report: SCI-FI Genre                                                                │
│                                                                                                                 │
│  💰 Average Box Office: $380M                                                                                   │
│  👥 Target Demographic: 18-40 years, 60% male                                                                   │
│  📅 Best Release Window: Summer and Holiday season                                                              │
│  🎬 Trending Themes:                                                                                            │
│     • AI & technology                                                                                           │
│     • Space exploration                                                                                         │
│     • Time travel                                                                                               │
│  📺 Streaming Performance: High - 88% completion rate                                                           │
│                                                                                                                 │
│  ✅ Data retrieved from Market Intelligence Database (MCP)                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Film Market Research Analyst                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│  Comprehensive Market Analysis for Sci-Fi Thriller Genre:                                                       │
│                                                                                                                 │
│  1. Current Market Performance:                                                                                 │
│     Sci-fi films continue to perform exceptionally well, averaging $380 million at the box office. This strong  │
│  financial performance underscores the genre's mainstream appeal and profitability.                             │
│                                                                                                                 │
│  2. Target Audience Insights:                                                                                   │
│     The core audience for sci-fi films is primarily aged 18 to 40 years, with a 60% male skew. This             │
│  demographic tends to be highly engaged with futuristic and technology-driven narratives, making them an ideal  │
│  target for sci-fi thrillers with complex and suspenseful storylines.                                           │
│                                                                                                                 │
│  3. Optimal Release Timing:                                                                                     │
│     The most advantageous release windows for sci-fi thrillers are summer and holiday seasons. These periods    │
│  coincide with higher audience availability and increased leisure time, maximizing box office potential.        │
│                                                                                                                 │
│  4. Strategic Recommendations Based on Trending Themes:                                                         │
│     Current popular themes in the sci-fi genre include:                                                         │
│     - Artificial Intelligence and advanced technology                                                           │
│     - Space exploration and adventure                                                                           │
│     - Time travel and its paradoxes                                                                             │
│                                                                                                                 │
│     Incorporating these themes into the narrative or marketing strategy will align the film with audience       │
│  interests and market trends. Additionally, the high streaming completion rate of 88% suggests that a combined  │
│  theatrical and streaming release strategy could effectively maximize audience reach and engagement.            │
│                                                                                                                 │
│  This market research supports strategic decisions for production and distribution, suggesting that sci-fi      │
│  thrillers like "Echoes of Tomorrow" should emphasize trending technological and time-based themes, target the  │
│  18-40 demographic, and aim for release during peak aud

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Conduct market research for the sci-fi thriller genre.                                                         │
│                                                                                                                 │
│      Use the MCP tool to fetch market data and provide:                                                         │
│      1. Current market performance of sci-fi films                                                              │
│      2. Target audience insights                                                                                │
│      3. Optimal release timing                                                                                  │
│      4. Strategic recommendations based on trending themes                                                      │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Film Market Research Analyst                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  8935cf36-49e7-448a-84b4-38ff43191ae6                                                                           │
│  Final Output: Final Answer:                                                                                    │
│                                                                                                                 │
│  Comprehensive Market Analysis for Sci-Fi Thriller Genre:                                                       │
│                                                                                                                 │
│  1. Current Market Performance:                                                                                 │
│     Sci-fi films continue to perform exceptionally well, averaging $380 million at the box office. This strong  │
│  financial performance underscores the genre's mainstream appeal and profitability.                             │
│                                                                                                                 │
│  2. Target Audience Insights:                                                                                   │
│     The core audience for sci-fi films is primarily aged 18 to 40 years, with a 60% male skew. This             │
│  demographic tends to be highly engaged with futuristic and technology-driven narratives, making them an ideal  │
│  target for sci-fi thrillers with complex and suspenseful storylines.                                           │
│                                                                                                                 │
│  3. Optimal Release Timing:                                                                                     │
│     The most advantageous release windows for sci-fi thrillers are summer and holiday seasons. These periods    │
│  coincide with higher audience availability and increased leisure time, maximizing box office potential.        │
│                                                                                                                 │
│  4. Strategic Recommendations Based on Trending Themes:                                                         │
│     Current popular themes in the sci-fi genre include:                                                         │
│     - Artificial Intelligence and advanced technology                                                           │
│     - Space exploration and adventure                                                                           │
│     - Time travel and its paradoxes                                                                             │
│                                                                                                                 │
│     Incorporating these themes into the narrative or marketing strategy will align the film with audience       │
│  interests and market trends. Additionally, the high streaming completion rate of 88% suggests that a combined  │
│  theatrical and streaming release strategy could effectively maximize audience reach and engagement.            │
│                                                                                                                 │
│  This market research supports strategic decisions for



✅ Analysis Complete!



## Step 9: View Final Results

Let's examine the consolidated output from all agents.

In [20]:
print("📋 FINAL PRODUCTION REPORT")
print("=" * 80)
print(result)
print("=" * 80)

📋 FINAL PRODUCTION REPORT
Final Answer:

Comprehensive Market Analysis for Sci-Fi Thriller Genre:

1. Current Market Performance:
   Sci-fi films continue to perform exceptionally well, averaging $380 million at the box office. This strong financial performance underscores the genre's mainstream appeal and profitability.

2. Target Audience Insights:
   The core audience for sci-fi films is primarily aged 18 to 40 years, with a 60% male skew. This demographic tends to be highly engaged with futuristic and technology-driven narratives, making them an ideal target for sci-fi thrillers with complex and suspenseful storylines.

3. Optimal Release Timing:
   The most advantageous release windows for sci-fi thrillers are summer and holiday seasons. These periods coincide with higher audience availability and increased leisure time, maximizing box office potential.

4. Strategic Recommendations Based on Trending Themes:
   Current popular themes in the sci-fi genre include:
   - Artificial In

## Step 10: Implementing Observability

Observability helps you monitor and debug your agents. Let's add logging and tracking capabilities.

**Key Observability Features:**
1. Execution time tracking
2. Tool usage monitoring
3. Error logging
4. Agent decision tracking

In [21]:
import time
import json
from datetime import datetime

class CrewObserver:
    """Custom observability class for monitoring CrewAI execution"""

    def __init__(self):
        self.logs = []
        self.start_time = None
        self.end_time = None

    def log_event(self, event_type: str, agent_name: str, details: dict):
        """Log an event during crew execution"""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'event_type': event_type,
            'agent': agent_name,
            'details': details
        }
        self.logs.append(log_entry)

    def start_monitoring(self):
        """Start monitoring crew execution"""
        self.start_time = time.time()
        self.log_event('CREW_START', 'System', {'message': 'Crew execution started'})

    def end_monitoring(self):
        """End monitoring and calculate metrics"""
        self.end_time = time.time()
        duration = self.end_time - self.start_time
        self.log_event('CREW_END', 'System', {
            'message': 'Crew execution completed',
            'duration_seconds': round(duration, 2)
        })

    def get_metrics(self):
        """Generate execution metrics"""
        if not self.end_time:
            return "Monitoring still in progress"

        duration = self.end_time - self.start_time
        agent_events = {}

        for log in self.logs:
            agent = log['agent']
            if agent != 'System':
                agent_events[agent] = agent_events.get(agent, 0) + 1

        metrics = f"""\n📊 EXECUTION METRICS\n{'='*60}
⏱️  Total Duration: {duration:.2f} seconds
📝 Total Events Logged: {len(self.logs)}
🤖 Agent Activity:"""

        for agent, count in agent_events.items():
            metrics += f"\n   • {agent}: {count} events"

        return metrics

    def display_logs(self, filter_agent=None):
        """Display execution logs"""
        print(f"\n📋 EXECUTION LOGS ({len(self.logs)} entries)")
        print("=" * 80)

        for log in self.logs:
            if filter_agent and log['agent'] != filter_agent:
                continue

            timestamp = log['timestamp'].split('T')[1].split('.')[0]
            print(f"[{timestamp}] {log['event_type']:15} | {log['agent']:25} | {log['details']}")

# Initialize observer
observer = CrewObserver()
print("✅ Observability system initialized")

✅ Observability system initialized


## Step 11: Run Crew with Observability

Now let's run our crew again with full observability enabled.

In [22]:
# Create a new task with observability hooks
observer.start_monitoring()

# Log agent initialization
observer.log_event('AGENT_INIT', 'Script Analyst', {'tools': ['sentiment_analyzer', 'soundtrack_recommender']})
observer.log_event('AGENT_INIT', 'Budget Manager', {'tools': ['budget_calculator']})
observer.log_event('AGENT_INIT', 'Market Researcher', {'tools': ['market_data_mcp']})

# New movie concept for second run
new_concept = """
Title: "The Last Symphony"
Genre: Drama
Logline: An elderly composer, losing her memory to dementia, races to complete her
final masterpiece while her estranged daughter learns to understand her through music.

Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across
the piano keys. The wonderful melody emerges from the fog of her fading memory, a
triumph over the terrible disease that threatens to steal her life's work."
"""

# Create new task
observed_task = Task(
    description=f"""Analyze this movie concept:
    {new_concept}

    Perform sentiment analysis and recommend soundtrack genres.""",
    expected_output="Sentiment analysis and soundtrack recommendations",
    agent=script_analyst,
)

# Log task start
observer.log_event('TASK_START', 'Script Analyst', {'task': 'Script analysis'})

# Create a smaller crew for demonstration
observed_crew = Crew(
    agents=[script_analyst],
    tasks=[observed_task],
    process=Process.sequential,
    verbose=1,  # Less verbose for cleaner output
)

print("🔍 Running crew with observability enabled...\n")

# Execute
result_observed = observed_crew.kickoff()

# Log completion
observer.log_event('TASK_COMPLETE', 'Script Analyst', {'status': 'success'})
observer.end_monitoring()

print("\n✅ Execution complete with observability tracking!")

🔍 Running crew with observability enabled...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  3f097e4f-3930-4ab6-ab0d-eb4748aac835                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this movie concept:                                                                              │
│                                                                                                                 │
│  Title: "The Last Symphony"                                                                                     │
│  Genre: Drama                                                                                                   │
│  Logline: An elderly composer, losing her memory to dementia, races to complete her                             │
│  final masterpiece while her estranged daughter learns to understand her through music.                         │
│                                                                                                                 │
│  Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across                      │
│  the piano keys. The wonderful melody emerges from the fog of her fading memory, a                              │
│  triumph over the terrible disease that threatens to steal her life's work."                                    │
│                                                                                                                 │
│                                                                                                                 │
│      Perform sentiment analysis and recommend soundtrack genres.                                                │
│  ID: a40d46dd-9e89-4cae-ad12-e1abc21cbcb6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Task: Analyze this movie concept:                                                                              │
│                                                                                                                 │
│  Title: "The Last Symphony"                                                                                     │
│  Genre: Drama                                                                                                   │
│  Logline: An elderly composer, losing her memory to dementia, races to complete her                             │
│  final masterpiece while her estranged daughter learns to understand her through music.                         │
│                                                                                                                 │
│  Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across                      │
│  the piano keys. The wonderful melody emerges from the fog of her fading memory, a                              │
│  triumph over the terrible disease that threatens to steal her life's work."                                    │
│                                                                                                                 │
│                                                                                                                 │
│      Perform sentiment analysis and recommend soundtrack genres.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool script_sentiment_analyzer executed with result: Sentiment Analysis Results:
- Positive tone: 3/5 (60.0%)
- Negative tone: 2/5 (40.0%)
- Suspenseful tone: 0/5 (0.0%)

Overall: Uplifting and positive story...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│  Sentiment Analysis:                                                                                            │
│  The script excerpt from "The Last Symphony" reflects an uplifting and positive emotional tone, with 60%        │
│  positive sentiment and 40% negative, capturing the bittersweet nature of Margaret’s struggle with dementia.    │
│  The mixture of joy and sadness effectively conveys the emotional depth of the story, highlighting themes of    │
│  memory loss, hope, and artistic triumph.                                                                       │
│                                                                                                                 │
│  Soundtrack Recommendations:                                                                                    │
│  For this emotional drama, the recommended soundtrack genres are Piano Solo, String Quartet, and Acoustic       │
│  Guitar. These intimate and acoustic genres complement the film’s themes of memory, music, and personal         │
│  connection. A combination of Piano Solo and Acoustic Guitar is suggested to provide dynamic emotional          │
│  resonance throughout the narrative, enhancing both tender and poignant moments.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Analyze this movie concept:                                                                                    │
│                                                                                                                 │
│  Title: "The Last Symphony"                                                                                     │
│  Genre: Drama                                                                                                   │
│  Logline: An elderly composer, losing her memory to dementia, races to complete her                             │
│  final masterpiece while her estranged daughter learns to understand her through music.                         │
│                                                                                                                 │
│  Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across                      │
│  the piano keys. The wonderful melody emerges from the fog of her fading memory, a                              │
│  triumph over the terrible disease that threatens to steal her life's work."                                    │
│                                                                                                                 │
│                                                                                                                 │
│      Perform sentiment analysis and recommend soundtrack genres.                                                │
│  Agent:                                                                                                         │
│  Script Analyst                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  3f097e4f-3930-4ab6-ab0d-eb4748aac835                                                                           │
│  Final Output: Final Answer:                                                                                    │
│                                                                                                                 │
│  Sentiment Analysis:                                                                                            │
│  The script excerpt from "The Last Symphony" reflects an uplifting and positive emotional tone, with 60%        │
│  positive sentiment and 40% negative, capturing the bittersweet nature of Margaret’s struggle with dementia.    │
│  The mixture of joy and sadness effectively conveys the emotional depth of the story, highlighting themes of    │
│  memory loss, hope, and artistic triumph.                                                                       │
│                                                                                                                 │
│  Soundtrack Recommendations:                                                                                    │
│  For this emotional drama, the recommended soundtrack genres are Piano Solo, String Quartet, and Acoustic       │
│  Guitar. These intimate and acoustic genres complement the film’s themes of memory, music, and personal         │
│  connection. A combination of Piano Solo and Acoustic Guitar is suggested to provide dynamic emotional          │
│  resonance throughout the narrative, enhancing both tender and poignant moments.                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Execution complete with observability tracking!


┌───────────────────────── Trace Batch Finalization ──────────────────────────┐
│ ✅ Trace batch finalized with session ID:                                   │
│ 8ee657be-7b5f-4e14-84ed-596bfb9aaeaf                                        │
│                                                                             │
│ 🔗 View here:                                                               │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/8ee657be-7b5f-4e │
│ 14-84ed-596bfb9aaeaf?access_code=TRACE-e578ba363f                           │
│ 🔑 Access Code: TRACE-e578ba363f                                            │
└─────────────────────────────────────────────────────────────────────────────┘


## 📈 Step 12: Analyze Observability Data

Let's examine the metrics and logs collected during execution.

In [23]:
# Display metrics
print(observer.get_metrics())

# Display all logs
observer.display_logs()

# Show result
print("\n📋 AGENT OUTPUT")
print("=" * 80)
print(result_observed)
print("=" * 80)


📊 EXECUTION METRICS
⏱️  Total Duration: 5.90 seconds
📝 Total Events Logged: 7
🤖 Agent Activity:
   • Script Analyst: 3 events
   • Budget Manager: 1 events
   • Market Researcher: 1 events

📋 EXECUTION LOGS (7 entries)
[14:40:51] CREW_START      | System                    | {'message': 'Crew execution started'}
[14:40:51] AGENT_INIT      | Script Analyst            | {'tools': ['sentiment_analyzer', 'soundtrack_recommender']}
[14:40:51] AGENT_INIT      | Budget Manager            | {'tools': ['budget_calculator']}
[14:40:51] AGENT_INIT      | Market Researcher         | {'tools': ['market_data_mcp']}
[14:40:51] TASK_START      | Script Analyst            | {'task': 'Script analysis'}
[14:40:57] TASK_COMPLETE   | Script Analyst            | {'status': 'success'}
[14:40:57] CREW_END        | System                    | {'message': 'Crew execution completed', 'duration_seconds': 5.9}

📋 AGENT OUTPUT
Final Answer:

Sentiment Analysis:
The script excerpt from "The Last Symphony" reflects 

## Step 13: Advanced Observability - Export Logs

In production, you'd want to export logs for analysis. Here's how to save them.

In [24]:
import json

# Export logs to JSON
def export_logs(observer, filename='crew_execution_logs.json'):
    """Export observability logs to a JSON file"""
    log_data = {
        'execution_summary': {
            'start_time': datetime.fromtimestamp(observer.start_time).isoformat(),
            'end_time': datetime.fromtimestamp(observer.end_time).isoformat(),
            'duration_seconds': round(observer.end_time - observer.start_time, 2),
            'total_events': len(observer.logs)
        },
        'events': observer.logs
    }

    with open(filename, 'w') as f:
        json.dump(log_data, f, indent=2)

    return f"✅ Logs exported to {filename}"

# Export the logs
print(export_logs(observer))

# Display a sample of the JSON structure
print("\n📄 Sample JSON structure:")
sample = {
    'execution_summary': 'metadata about the run',
    'events': [
        {
            'timestamp': '2025-11-30T10:30:45.123456',
            'event_type': 'TASK_START',
            'agent': 'Script Analyst',
            'details': {'task': 'Script analysis'}
        }
    ]
}
print(json.dumps(sample, indent=2))

✅ Logs exported to crew_execution_logs.json

📄 Sample JSON structure:
{
  "execution_summary": "metadata about the run",
  "events": [
    {
      "timestamp": "2025-11-30T10:30:45.123456",
      "event_type": "TASK_START",
      "agent": "Script Analyst",
      "details": {
        "task": "Script analysis"
      }
    }
  ]
}


## 🎯 CHALLENGE: Build Your Own Movie Production Crew!

Now it's your turn to apply what you've learned! Complete the following challenge:

### 🎬 Challenge Description

Create a **Movie Casting Assistant** crew that helps with casting decisions for a new film.

### Requirements:

1. **Create TWO custom tools:**
   - `actor_availability_checker`: Takes an actor name and date range, returns if they're available (simulate with logic)
   - `role_matcher`: Takes character description and returns suggested actor traits (age range, experience level, acting style)

2. **Create ONE MCP-style tool:**
   - `fetch_actor_database`: Simulates querying a database of actors with their filmography, awards, and availability

3. **Create TWO agents:**
   - **Casting Director**: Uses role_matcher and fetch_actor_database
   - **Schedule Coordinator**: Uses actor_availability_checker

4. **Define tasks:**
   - Task 1: Find suitable actors for a lead role (give a character description)
   - Task 2: Check availability of suggested actors for specific shooting dates

5. **Add observability:**
   - Use the CrewObserver to track execution
   - Export logs at the end

### 💡 Bonus Points:
- Make your character descriptions creative and unique
- Add personality to your agents' backstories
- Include error handling in your tools
- Create a visualization of the observability data

### Sample Character (feel free to create your own!):
```
Character: Captain Zara Quinn
Description: A 35-year-old space captain with a mysterious past, known for her
quick wit and exceptional piloting skills. Must convey both strength and vulnerability.
```

### 📝 Your Code Goes Below:

Good luck! 🚀

In [26]:
# MOVIE CASTING ASSISTANT CREW - Complete Implementation

# Step 1: Create custom tools for casting
from datetime import datetime, timedelta
import random

@tool("Actor Availability Checker")
def actor_availability_checker(actor_name: str, start_date: str, end_date: str) -> str:
    """
    Checks if an actor is available for a specific date range.
    
    Args:
        actor_name: Name of the actor
        start_date: Start date in format YYYY-MM-DD
        end_date: End date in format YYYY-MM-DD
    
    Returns:
        Availability status with reasoning
    """
    # Simulated actor schedules
    actor_schedules = {
        'Emma Stone': {'busy': [('2025-11-01', '2025-12-15'), ('2026-03-01', '2026-04-30')]},
        'Oscar Isaac': {'busy': [('2025-10-01', '2025-11-30'), ('2026-02-15', '2026-03-30')]},
        'Timothée Chalamet': {'busy': [('2025-12-01', '2026-02-28')]},
        'Florence Pugh': {'busy': [('2025-11-15', '2026-01-15')]},
        'Dev Patel': {'busy': [('2026-01-01', '2026-02-01')]},
    }
    
    try:
        start = datetime.strptime(start_date, '%Y-%m-%d')
        end = datetime.strptime(end_date, '%Y-%m-%d')
        
        if start > end:
            return f"❌ Invalid date range: Start date cannot be after end date"
        
        actor_key = next((key for key in actor_schedules if key.lower() == actor_name.lower()), None)
        
        if not actor_key:
            return f"⚠️ Actor '{actor_name}' not found in database. Assuming available for speculation."
        
        busy_periods = actor_schedules[actor_key]['busy']
        
        for busy_start, busy_end in busy_periods:
            busy_start_dt = datetime.strptime(busy_start, '%Y-%m-%d')
            busy_end_dt = datetime.strptime(busy_end, '%Y-%m-%d')
            
            if start <= busy_end_dt and end >= busy_start_dt:
                return f"❌ {actor_key} is UNAVAILABLE\n   Conflict: Busy from {busy_start} to {busy_end}"
        
        return f"✅ {actor_key} is AVAILABLE for {start_date} to {end_date}\n   Perfect timing for this production!"
    
    except ValueError as e:
        return f"❌ Date format error: Use YYYY-MM-DD format. {str(e)}"

@tool("Role Matcher")
def role_matcher(character_description: str) -> str:
    """
    Suggests ideal actor traits based on character description.
    
    Args:
        character_description: Detailed description of the character
    
    Returns:
        Suggested actor traits and profile
    """
    # Analyze character description for keywords
    description_lower = character_description.lower()
    
    # Character trait analysis
    analysis = {
        'age_range': None,
        'acting_style': [],
        'experience_level': None,
        'physical_traits': [],
        'emotional_depth': None,
    }
    
    # Age detection
    if any(word in description_lower for word in ['young', 'teenager', '20s', '30s']):
        if 'teenager' in description_lower or '20s' in description_lower:
            analysis['age_range'] = '18-30 years'
        else:
            analysis['age_range'] = '25-40 years'
    elif any(word in description_lower for word in ['elderly', 'old', 'aged', '60', '70']):
        analysis['age_range'] = '55+ years'
    else:
        analysis['age_range'] = '30-50 years (mature)'
    
    # Acting style detection
    if any(word in description_lower for word in ['intense', 'dramatic', 'serious']):
        analysis['acting_style'].append('Dramatic/Intense')
    if any(word in description_lower for word in ['witty', 'charming', 'humorous']):
        analysis['acting_style'].append('Charismatic/Charming')
    if any(word in description_lower for word in ['mysterious', 'enigmatic', 'dark']):
        analysis['acting_style'].append('Mysterious/Complex')
    if any(word in description_lower for word in ['vulnerable', 'emotional', 'deep']):
        analysis['acting_style'].append('Deeply Emotional')
    
    # Experience level
    if any(word in description_lower for word in ['legendary', 'master', 'veteran', 'experienced']):
        analysis['experience_level'] = 'Veteran (15+ years)'
    elif any(word in description_lower for word in ['seasoned', 'proven']):
        analysis['experience_level'] = 'Established (5-15 years)'
    else:
        analysis['experience_level'] = 'Mid-range (2-8 years)'
    
    # Build response
    result = f"🎭 ROLE MATCHING PROFILE:\n\n"
    result += f"📊 Ideal Actor Specifications:\n"
    result += f"   • Age Range: {analysis['age_range']}\n"
    result += f"   • Experience: {analysis['experience_level']}\n"
    result += f"   • Acting Styles: {', '.join(analysis['acting_style']) if analysis['acting_style'] else 'Versatile'}\n"
    result += f"\n💡 Casting Tips:\n"
    result += f"   • Look for actors with range in {analysis['acting_style'][0] if analysis['acting_style'] else 'dramatic'} roles\n"
    result += f"   • Consider method actors for emotionally demanding scenes\n"
    result += f"   • Screen test multiple candidates within age range\n"
    
    return result

# Step 2: Create MCP-style tool for actor database
@tool("Fetch Actor Database")
def fetch_actor_database(search_criteria: str) -> str:
    """
    Simulates querying an external actor database via MCP.
    Returns actor profiles matching the criteria.
    
    Args:
        search_criteria: Search criteria (genre, age range, awards, etc.)
    
    Returns:
        List of matching actors with their information
    """
    # Simulated actor database
    actor_database = {
        'Emma Stone': {
            'age': 36,
            'filmography': ['La La Land', 'The Favourite', 'Kinds of Kindness', 'Poor Things'],
            'awards': ['Academy Award', "Golden Globe for Best Actress"],
            'last_available': '2026-05-01',
            'rate_per_day': '$500,000',
        },
        'Oscar Isaac': {
            'age': 46,
            'filmography': ['Dune', 'Moon Knight', 'Ex Machina', 'Scenes from a Marriage'],
            'awards': ['Golden Globe', 'BAFTA nomination'],
            'last_available': '2026-04-15',
            'rate_per_day': '$450,000',
        },
        'Timothée Chalamet': {
            'age': 29,
            'filmography': ['Dune', 'The King', 'Interstellar', 'Call Me By Your Name'],
            'awards': ['BAFTA nomination', 'Golden Globe nomination'],
            'last_available': '2026-03-01',
            'rate_per_day': '$400,000',
        },
        'Florence Pugh': {
            'age': 28,
            'filmography': ['Midsommar', 'Little Women', 'Oppenheimer', 'Don\'t Worry Darling'],
            'awards': ['BAFTA nomination'],
            'last_available': '2026-02-01',
            'rate_per_day': '$350,000',
        },
        'Dev Patel': {
            'age': 34,
            'filmography': ['Monkey Man', 'The Green Knight', 'Hotel Mumbai', 'Slumdog Millionaire'],
            'awards': ['Golden Globe', 'Independent Spirit Award'],
            'last_available': '2026-05-15',
            'rate_per_day': '$380,000',
        },
    }
    
    result = f"🎬 ACTOR DATABASE SEARCH RESULTS\n"
    result += f"Search Criteria: {search_criteria}\n"
    result += f"{'='*70}\n\n"
    
    for actor_name, info in actor_database.items():
        result += f"👤 {actor_name}\n"
        result += f"   📅 Age: {info['age']}\n"
        result += f"   🎥 Recent Films: {', '.join(info['filmography'][-2:])}\n"
        result += f"   🏆 Awards: {', '.join(info['awards'][:1])}\n"
        result += f"   💰 Rate: {info['rate_per_day']} per day\n"
        result += f"   ✓ Next Available: {info['last_available']}\n\n"
    
    return result

# Step 3: Create agents with personality
casting_director = Agent(
    role="Casting Director",
    goal="Find the perfect actors for each role by analyzing character requirements and matching them with ideal candidates",
    backstory="""You are a legendary casting director with 25 years of experience in Hollywood.
    You've cast iconic roles and have an uncanny ability to spot talent and predict on-screen chemistry.
    Your instincts have led to Oscar-winning performances. You approach each role with passion and precision,
    believing that casting is 80% of what makes a great film. You're meticulous, creative, and always think outside the box.""",
    tools=[role_matcher, fetch_actor_database],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

schedule_coordinator = Agent(
    role="Schedule Coordinator",
    goal="Ensure all cast members are available during production dates and optimize shooting schedule",
    backstory="""You are a detail-oriented production scheduler with 12 years of experience managing complex filming schedules.
    You excel at logistics and problem-solving, often finding creative solutions when conflicts arise.
    Your nickname is 'The Miracle Worker' because you always make impossible schedules work.
    You understand the nuances of actor availability, travel time, and production constraints.""",
    tools=[actor_availability_checker],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Created 2 specialized casting agents:")
print("   1. Casting Director (role matching)")
print("   2. Schedule Coordinator (availability checking)")

# Step 4: Define casting tasks with creative character
character_description = """
Character: Lyra Vex
A 32-year-old rogue astrophysicist with a tortured past and brilliant mind.
She's mysterious, haunted by secrets, yet possesses unmistakable charm and wit.
She leads with emotional intelligence but hides deep vulnerability beneath a cool exterior.
The role demands someone who can portray both strength and fragility seamlessly.
Must convey genius-level intellect without coming across as arrogant."""

task_find_actor = Task(
    description=f"""Your first task is to find the perfect actor for this lead role:

CHARACTER PROFILE:
{character_description}

Use the role_matcher to analyze the character requirements and fetch_actor_database to find suitable candidates.
Provide 2-3 specific actor recommendations with justification for why they'd be perfect for this role.
Consider their filmography, awards, and acting range.""",
    expected_output="Actor recommendations with detailed casting analysis and justification",
    agent=casting_director,
)

task_check_availability = Task(
    description="""Now that we have casting recommendations, check availability for:

Shooting Dates: 2026-04-15 to 2026-07-30 (16 weeks of production)

Check availability for these recommended actors:
- Emma Stone
- Oscar Isaac
- Dev Patel

Identify any scheduling conflicts and suggest solutions.
Provide alternative dates if conflicts exist.""",
    expected_output="Availability report for all candidates with conflict resolution strategies",
    agent=schedule_coordinator,
)

print("✅ Created 2 casting tasks:")
print("   1. Actor recommendation task")
print("   2. Availability verification task")

# Step 5: Set up observability
casting_observer = CrewObserver()

# Step 6: Run the casting crew
casting_observer.start_monitoring()

# Log initialization
casting_observer.log_event('RESOURCE_INIT', 'Casting Director', {
    'tools': ['role_matcher', 'fetch_actor_database'],
    'specialty': 'Finding perfect actor matches'
})
casting_observer.log_event('RESOURCE_INIT', 'Schedule Coordinator', {
    'tools': ['actor_availability_checker'],
    'specialty': 'Managing production schedule'
})

# Create and execute the casting crew
casting_crew = Crew(
    agents=[casting_director, schedule_coordinator],
    tasks=[task_find_actor, task_check_availability],
    process=Process.sequential,
    verbose=True,
)

print("\n🎬 CASTING CREW EXECUTION STARTING...\n")
print("=" * 80)

casting_observer.log_event('CREW_START', 'Casting Crew', {'mission': 'Find and schedule ideal actors'})

casting_result = casting_crew.kickoff()

casting_observer.log_event('CREW_COMPLETE', 'Casting Crew', {'status': 'success'})
casting_observer.end_monitoring()

print("=" * 80)
print("\n✅ CASTING PROCESS COMPLETE!\n")

# Step 7: Display results and metrics
print("📋 CASTING RESULTS")
print("=" * 80)
print(casting_result)
print("=" * 80)

# Display metrics
print(casting_observer.get_metrics())

# Display detailed logs
casting_observer.display_logs()

# Export logs to file
print("\n" + export_logs(casting_observer, 'casting_crew_logs.json'))

# Bonus: Create visualization data
print("\n📊 OBSERVABILITY DATA EXPORT")
print("=" * 80)
log_summary = {
    'production': 'Captain Lyra Vex - Lead Role',
    'total_events': len(casting_observer.logs),
    'execution_time': round(casting_observer.end_time - casting_observer.start_time, 2),
    'agents_involved': 2,
    'tools_used': 5,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(log_summary, indent=2))

✅ Created 2 specialized casting agents:
   1. Casting Director (role matching)
   2. Schedule Coordinator (availability checking)
✅ Created 2 casting tasks:
   1. Actor recommendation task
   2. Availability verification task

🎬 CASTING CREW EXECUTION STARTING...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  7461db13-e8be-4586-ae7a-04c3d572d125                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Your first task is to find the perfect actor for this lead role:                                         │
│                                                                                                                 │
│  CHARACTER PROFILE:                                                                                             │
│                                                                                                                 │
│  Character: Lyra Vex                                                                                            │
│  A 32-year-old rogue astrophysicist with a tortured past and brilliant mind.                                    │
│  She's mysterious, haunted by secrets, yet possesses unmistakable charm and wit.                                │
│  She leads with emotional intelligence but hides deep vulnerability beneath a cool exterior.                    │
│  The role demands someone who can portray both strength and fragility seamlessly.                               │
│  Must convey genius-level intellect without coming across as arrogant.                                          │
│                                                                                                                 │
│  Use the role_matcher to analyze the character requirements and fetch_actor_database to find suitable           │
│  candidates.                                                                                                    │
│  Provide 2-3 specific actor recommendations with justification for why they'd be perfect for this role.         │
│  Consider their filmography, awards, and acting range.                                                          │
│  ID: ae2562d0-eaca-426d-8ee8-14d041293901                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Casting Director                                                                                        │
│                                                                                                                 │
│  Task: Your first task is to find the perfect actor for this lead role:                                         │
│                                                                                                                 │
│  CHARACTER PROFILE:                                                                                             │
│                                                                                                                 │
│  Character: Lyra Vex                                                                                            │
│  A 32-year-old rogue astrophysicist with a tortured past and brilliant mind.                                    │
│  She's mysterious, haunted by secrets, yet possesses unmistakable charm and wit.                                │
│  She leads with emotional intelligence but hides deep vulnerability beneath a cool exterior.                    │
│  The role demands someone who can portray both strength and fragility seamlessly.                               │
│  Must convey genius-level intellect without coming across as arrogant.                                          │
│                                                                                                                 │
│  Use the role_matcher to analyze the character requirements and fetch_actor_database to find suitable           │
│  candidates.                                                                                                    │
│  Provide 2-3 specific actor recommendations with justification for why they'd be perfect for this role.         │
│  Consider their filmography, awards, and acting range.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool role_matcher executed with result: 🎭 ROLE MATCHING PROFILE:

📊 Ideal Actor Specifications:
   • Age Range: 55+ years
   • Experience: Mid-range (2-8 years)
   • Acting Styles: Mysterious/Complex, Deeply Emotional

💡 Casting Tips:
   • ...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Schedule Coordinator                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Availability Report:                                                                                           │
│                                                                                                                 │
│  1. Emma Stone:                                                                                                 │
│     - Unavailable for the entire shooting period (2026-04-15 to 2026-07-30).                                    │
│     - Conflict: She is busy from 2026-03-01 to 2026-04-30.                                                      │
│     - Suggested Solutions:                                                                                      │
│       - Delay the start of production to after 2026-05-01, allowing Emma Stone to join from that date onward,   │
│  if the schedule can be adjusted.                                                                               │
│       - Alternatively, consider splitting Emma's shooting schedule to start after 2026-05-01 while other        │
│  scenes with available actors are shot earlier.                                                                 │
│       - If neither is feasible, consider casting an alternative actor for Emma Stone's role.                    │
│                                                                                                                 │
│  2. Oscar Isaac:                                                                                                │
│     - Fully available for the entire shooting period (2026-04-15 to 2026-07-30).                                │
│     - No conflicts.                                                                                             │
│                                                                                                                 │
│  3. Dev Patel:                                                                                                  │
│     - Fully available for the entire shooting period (2026-04-15 to 2026-07-30).                                │
│     - No conflicts.                                                                                             │
│                                                                                                                 │
│  Summary:                                                                                                       │
│  Oscar Isaac and Dev Patel have no scheduling conflicts and can be fully integrated into the 16-week shooting   │
│  schedule. Emma Stone's current commitments present a conflict during the first two weeks of production.        │
│  Adjusting the schedule to start after 2026-05-01 or rearranging shooting sequences could resolve this. If      │
│  schedule adjustments are not possible, alternative casting for Emma Stone's role should be considered.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Now that we have casting recommendations, check availability for:                                              │
│                                                                                                                 │
│  Shooting Dates: 2026-04-15 to 2026-07-30 (16 weeks of production)                                              │
│                                                                                                                 │
│  Check availability for these recommended actors:                                                               │
│  - Emma Stone                                                                                                   │
│  - Oscar Isaac                                                                                                  │
│  - Dev Patel                                                                                                    │
│                                                                                                                 │
│  Identify any scheduling conflicts and suggest solutions.                                                       │
│  Provide alternative dates if conflicts exist.                                                                  │
│  Agent:                                                                                                         │
│  Schedule Coordinator                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  7461db13-e8be-4586-ae7a-04c3d572d125                                                                           │
│  Final Output: Availability Report:                                                                             │
│                                                                                                                 │
│  1. Emma Stone:                                                                                                 │
│     - Unavailable for the entire shooting period (2026-04-15 to 2026-07-30).                                    │
│     - Conflict: She is busy from 2026-03-01 to 2026-04-30.                                                      │
│     - Suggested Solutions:                                                                                      │
│       - Delay the start of production to after 2026-05-01, allowing Emma Stone to join from that date onward,   │
│  if the schedule can be adjusted.                                                                               │
│       - Alternatively, consider splitting Emma's shooting schedule to start after 2026-05-01 while other        │
│  scenes with available actors are shot earlier.                                                                 │
│       - If neither is feasible, consider casting an alternative actor for Emma Stone's role.                    │
│                                                                                                                 │
│  2. Oscar Isaac:                                                                                                │
│     - Fully available for the entire shooting period (2026-04-15 to 2026-07-30).                                │
│     - No conflicts.                                                                                             │
│                                                                                                                 │
│  3. Dev Patel:                                                                                                  │
│     - Fully available for the entire shooting period (2026-04-15 to 2026-07-30).                                │
│     - No conflicts.                                                                                             │
│                                                                                                                 │
│  Summary:                                                                                                       │
│  Oscar Isaac and Dev Patel have no scheduling conflicts and can be fully integrated into the 16-week shooting   │
│  schedule. Emma Stone's current commitments present a conflict during the first two weeks of production.        │
│  Adjusting the schedule to start after 2026-05-01 or rearranging shooting sequences could resolve this. If      │
│  schedule adjustments are not possible, alternative casting for Emma Stone's role should be considered.         │
│                                                                                                                 │
│                                                                                                                 │
╰───────────────────────────────────────────────────────


✅ CASTING PROCESS COMPLETE!

📋 CASTING RESULTS
Availability Report:

1. Emma Stone:
   - Unavailable for the entire shooting period (2026-04-15 to 2026-07-30).
   - Conflict: She is busy from 2026-03-01 to 2026-04-30.
   - Suggested Solutions:
     - Delay the start of production to after 2026-05-01, allowing Emma Stone to join from that date onward, if the schedule can be adjusted.
     - Alternatively, consider splitting Emma's shooting schedule to start after 2026-05-01 while other scenes with available actors are shot earlier.
     - If neither is feasible, consider casting an alternative actor for Emma Stone's role.

2. Oscar Isaac:
   - Fully available for the entire shooting period (2026-04-15 to 2026-07-30).
   - No conflicts.

3. Dev Patel:
   - Fully available for the entire shooting period (2026-04-15 to 2026-07-30).
   - No conflicts.

Summary:
Oscar Isaac and Dev Patel have no scheduling conflicts and can be fully integrated into the 16-week shooting schedule. Emma Stone'

┌───────────────────────── Trace Batch Finalization ──────────────────────────┐
│ ✅ Trace batch finalized with session ID:                                   │
│ e31fe944-0a2f-4b50-8274-286d3364b3dc                                        │
│                                                                             │
│ 🔗 View here:                                                               │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/e31fe944-0a2f-4b │
│ 50-8274-286d3364b3dc?access_code=TRACE-b2a4263548                           │
│ 🔑 Access Code: TRACE-b2a4263548                                            │
└─────────────────────────────────────────────────────────────────────────────┘


## 🎉 Congratulations!

You've completed the CrewAI practice notebook! You've learned:

✅ How to create custom tools for agents  
✅ How to implement MCP-style tools for external integrations  
✅ How to configure and use Gemini 2.5 Flash model  
✅ How to create specialized agents with specific roles  
✅ How to define and orchestrate tasks  
✅ How to implement observability and monitoring  
✅ How to export and analyze execution logs  

### 🔗 Useful Resources:

- [CrewAI Documentation](https://docs.crewai.com/)
- [Google AI Studio](https://makersuite.google.com/)
- [LangChain Tools](https://python.langchain.com/docs/modules/agents/tools/)

Happy coding! 🚀🎬